# Moshi Compression — Session S2..S4 (Phase 0 full teacher cache, sharded)

**One notebook, three runs.** Set `SHARD_IDX` in Cell 5 to 0, 1, or 2 and run
end-to-end. Each run produces ~20 000 cached windows (≈73 GB tar) and a
separate Kaggle dataset.

## Carry-over from S1

| Knob | Decision (S1 evidence) |
|---|---|
| Teacher load | layers 0–15 cuda:0, 16–31 + heads cuda:1 (verbatim) |
| Patches | `torch.compile` off, `CUDAGraphed` no-op (verbatim) |
| Throughput | 171.8 windows/min (≈347 ms/window, very stable) |
| Per-window payload | hidden fp16 + text top-256 idx int32 + val fp16 = **3.65 MB** real |
| Semantic logits | **dropped** — Depformer is streaming, can't run T=375 in one shot. Re-cache at Phase 4. |
| Audio source | LibriSpeech `train-clean-360` + `train-other-500` (860 h, no repeats) |

## What is NEW in S2

1. **Shard-aware keys**: `b"s{SHARD_IDX}_{idx:06d}"` so Phase 1 can read across all 3 datasets without collision.
2. **`map_size = 80 GB`** (S1 used 30 GB and we needed 73). Sparse file, costs nothing on disk until written.
3. **In-loop compact-and-tar every 5 000 windows** — if Kaggle kills the kernel mid-session, we still have a partial tar to upload.
4. **HF streaming with skip-N** so each shard sees a different audio range.
5. **Upload via `--dir-mode tar`** — no more "Skipping folder" silent loss.

## Per-session timing budget (9 h Kaggle session)

| Phase | Time |
|---|---|
| Setup (env + installs + teacher load) | ~30 min |
| Cache loop (20 k windows @ 171.8 w/min) | ~117 min |
| Periodic compacts (3× compact+tar of growing LMDB) | ~10 min |
| Final compact + tar (~73 GB) | ~10 min |
| Kaggle dataset upload (~73 GB tar) | ~30 min |
| **Total** | **≈3.3 h** ← well under budget |

We could push to 30 k/session but then upload becomes the long pole and a
crash near the end loses 30 k windows of work instead of 20 k.

## Exit criteria per shard run

* `kaggle.com/<user>/moshi-teacher-cache-shard{N}` exists with `pilot.lmdb.tar` ≥ 60 GB.
* `S2_throughput_shard{N}.json` records final w/min and any failed-window indices.
* `MANIFEST.md` records the LibriSpeech item-index range covered by this shard.


## Cell 1 — Global patches (verbatim from S1)


In [1]:
# Disable torch.compile globally.
# Kaggle T4 Inductor occasionally emits bf16 intrinsics → "no kernel image" crash.
import os
import torch

os.environ["TORCH_COMPILE_DISABLE"] = "1"
torch._dynamo.config.disable = True
print("torch.compile disabled globally")
print("NOTE: CUDAGraphed patch will be applied in Cell 3b, after moshi is installed.")


torch.compile disabled globally
NOTE: CUDAGraphed patch will be applied in Cell 3b, after moshi is installed.


## Cell 2 — Environment verification (verbatim from S1)


In [2]:
import sys
print("python :", sys.version)
print("torch  :", torch.__version__, "  cuda:", torch.version.cuda)
print("cuda available  :", torch.cuda.is_available())
print("device count    :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, sm {p.major}.{p.minor}, "
          f"total {p.total_memory / 1e9:.1f} GB")

assert torch.cuda.device_count() == 2, (
    f"Need dual-GPU Kaggle runtime, got {torch.cuda.device_count()} GPU(s)")

for i in range(2):
    p = torch.cuda.get_device_properties(i)
    assert (p.major, p.minor) == (7, 5), (
        f"Expected T4 (sm_75) on cuda:{i}, got sm_{p.major}.{p.minor}")
    assert p.total_memory >= 15_000_000_000, (
        f"cuda:{i} reports only {p.total_memory/1e9:.1f} GB, expected ~16 GB")

def hw_bf16_supported():
    return all(
        torch.cuda.get_device_capability(i)[0] >= 8
        for i in range(torch.cuda.device_count())
    )

assert not hw_bf16_supported(), (
    "Hardware bf16 detected (CC >= 8.0) — are you on a non-T4 GPU?")

torch.cuda.set_device(0)
print("=== environment check PASSED ===")


python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch  : 2.10.0+cu128   cuda: 12.8
cuda available  : True
device count    : 2
  cuda:0 = Tesla T4, sm 7.5, total 15.6 GB
  cuda:1 = Tesla T4, sm 7.5, total 15.6 GB
=== environment check PASSED ===


## Cell 3 — Pinned installs (S1 + nothing new)

`tar` is BSD/GNU and already on the Kaggle base image, so no extra install.


In [3]:
import subprocess, sys, os

for pkg in [
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "bitsandbytes>=0.45,<0.50",
    "sentencepiece",
    "einops",
    "lmdb",
    "soundfile",
    "librosa",
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

MOSHI_SRC = next(
    (p for p in [
        "/kaggle/input/datasets/mhassann/moshi-repo/moshi/moshi",
        "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi",
        "/kaggle/input/moshi-repo/moshi/moshi",
    ] if __import__("pathlib").Path(p).exists()),
    "/kaggle/input/datasets/mhassann/moshi-repo/moshi/moshi"
)
MOSHI_DST = "/kaggle/working/moshi_repo"

if not os.path.exists(MOSHI_DST):
    print(f"Copying moshi repo to {MOSHI_DST} …")
    ret = subprocess.run(["cp", "-r", MOSHI_SRC, MOSHI_DST], capture_output=True, text=True)
    if ret.returncode != 0:
        raise RuntimeError(f"cp failed:\n{ret.stderr}")
    print("Copy done")
else:
    print(f"Repo already at {MOSHI_DST}")

print("Installing moshi (editable) …")
ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", MOSHI_DST],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print("pip stdout:", ret.stdout)
    print("pip stderr:", ret.stderr)
    raise RuntimeError("moshi editable install failed")
print("moshi installed (editable)")

import site, importlib
site.addsitedir(site.getsitepackages()[0])
if MOSHI_DST not in sys.path:
    sys.path.insert(0, MOSHI_DST)
importlib.invalidate_caches()

import moshi, transformers, bitsandbytes as bnb, lmdb, soundfile, librosa
print(f"moshi from: {moshi.__file__}")
print("transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("lmdb         :", lmdb.__version__)
print("=== installs OK ===")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 93.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 17.4 MB/s eta 0:00:00
Copying moshi repo to /kaggle/working/moshi_repo …
Copy done
Installing moshi (editable) …
moshi installed (editable)
moshi from: /kaggle/working/moshi_repo/moshi/__init__.py
transformers: 4.44.2
bitsandbytes: 0.49.2
lmdb         : 2.2.0
=== installs OK ===


In [4]:
import moshi.utils.compile as _moshi_compile

class _NoGraph:
    def __init__(self, fn, *a, **kw): self.fn = fn
    def __call__(self, *a, **kw):    return self.fn(*a, **kw)

_moshi_compile.CUDAGraphed = _NoGraph
print("CUDAGraphed monkey-patched to no-op")

import torch
assert torch._dynamo.config.disable
print("torch.compile still disabled — both patches active")


CUDAGraphed monkey-patched to no-op
torch.compile still disabled — both patches active


## Cell 4 — Load teacher + Mimi + shard (verbatim from S1)

Same memory layout: layers 0–15 on cuda:0 (~7.7 GB), 16–31 + heads on cuda:1
(~8.6 GB), Mimi on cuda:0 fp16. Teacher in `eval()` + frozen.


In [5]:
import torch, pathlib, time, shutil
from moshi.models.loaders import CheckpointInfo

REPO_ID  = "kyutai/moshiko-pytorch-bf16"

# Priority order for weight location:
#  1. /kaggle/working/moshiko-weights  — persists across kernel restarts in same session
#  2. /tmp/moshiko-weights             — fast but wiped on restart
# We always prefer /kaggle/working if the files are already there.
# Always use /tmp for weights — /kaggle/working disk is needed for the LMDB tar.
# /tmp has ~1.2 TB free on Kaggle. Weights are deleted after loading into RAM.
LOAD_DIR = pathlib.Path("/tmp/moshiko-weights")
LOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f"Weight cache dir: {LOAD_DIR}")

MOSHI_FILE = "model.safetensors"
MIMI_FILE  = "tokenizer-e351c8d8-checkpoint125.safetensors"
TOK_FILE   = "tokenizer_spm_32k_3.model"
FILES = {
    MOSHI_FILE : 14_000_000_000,
    MIMI_FILE  : 350_000_000,
    TOK_FILE   : 500_000,
}

def already_done(filename, min_size):
    p = LOAD_DIR / filename
    return p.exists() and p.stat().st_size >= min_size

from huggingface_hub import hf_hub_download

if not all(already_done(f, s) for f, s in FILES.items()):
    for filename, min_size in FILES.items():
        if already_done(filename, min_size):
            print(f"SKIP {filename}")
            continue
        attempt = 0
        while True:
            attempt += 1
            print(f"[attempt {attempt}] {filename}")
            try:
                hf_hub_download(repo_id=REPO_ID, filename=filename,
                                local_dir=str(LOAD_DIR), force_download=False)
                if already_done(filename, min_size):
                    print("  OK"); break
            except Exception as e:
                print(f"  Error: {e} — retry in 10s"); time.sleep(10)
else:
    print("All files already in /tmp")

moshi_weights = LOAD_DIR / MOSHI_FILE
mimi_weights  = LOAD_DIR / MIMI_FILE
tokenizer     = LOAD_DIR / TOK_FILE

info = CheckpointInfo(moshi_weights=moshi_weights, mimi_weights=mimi_weights,
                      tokenizer=tokenizer, lm_config=None)

print("\nLoading teacher LM to CPU …")
teacher_lm = info.get_moshi(device="cpu", dtype=torch.float16)
print("Loading Mimi to CPU …")
mimi = info.get_mimi(device="cpu")

# Delete weights from /tmp — already loaded into RAM, disk not needed
if str(LOAD_DIR).startswith("/tmp"):
    import shutil
    shutil.rmtree(LOAD_DIR)
    print(f"Deleted {LOAD_DIR} — weights are in RAM")

torch.cuda.empty_cache()
print("\nSharding teacher layer-by-layer …")
teacher_lm.emb.to("cuda:0")
teacher_lm.text_emb.to("cuda:0")

SPLIT = 16
for layer in teacher_lm.transformer.layers[:SPLIT]:
    layer.to("cuda:0")
for layer in teacher_lm.transformer.layers[SPLIT:]:
    layer.to("cuda:1")

for attr in ["out_norm", "text_linear", "depformer_in", "depformer",
             "depformer_emb", "depformer_text_emb", "linears"]:
    if hasattr(teacher_lm, attr):
        getattr(teacher_lm, attr).to("cuda:1")

mimi = mimi.to(device="cuda:0", dtype=torch.float16)

torch.cuda.synchronize()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  cuda:{i} after shard: free {free/1e9:.2f} / total {total/1e9:.2f} GB")

_orig_layers = teacher_lm.transformer.layers
def _sharded_forward(x, *args, **kwargs):
    for layer in _orig_layers[:SPLIT]:
        x = layer(x)
    x = x.to("cuda:1", non_blocking=True)
    for layer in _orig_layers[SPLIT:]:
        x = layer(x)
    return x
teacher_lm.transformer.forward = _sharded_forward

teacher_lm.eval()
for p in teacher_lm.parameters():
    p.requires_grad_(False)
print("Teacher set to eval() and all params frozen")

with torch.inference_mode():
    dummy = torch.zeros(1, teacher_lm.num_codebooks, 8,
                        dtype=torch.long, device="cuda:0")
    _ = teacher_lm.forward_text(dummy)
print("Teacher warmup OK")
print("\n=== Cell 4 PASSED ===")


Weight cache dir: /tmp/moshiko-weights
[attempt 1] model.safetensors


model.safetensors:   0%|          | 0.00/15.4G [00:00<?, ?B/s]

  OK
[attempt 1] tokenizer-e351c8d8-checkpoint125.safetensors


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

  OK
[attempt 1] tokenizer_spm_32k_3.model


tokenizer_spm_32k_3.model:   0%|          | 0.00/553k [00:00<?, ?B/s]

  OK

Loading teacher LM to CPU …
Loading Mimi to CPU …
Deleted /tmp/moshiko-weights — weights are in RAM

Sharding teacher layer-by-layer …
  cuda:0 after shard: free 8.21 / total 15.64 GB
  cuda:1 after shard: free 7.13 / total 15.64 GB
Teacher set to eval() and all params frozen
Teacher warmup OK

=== Cell 4 PASSED ===


## Cell 5 — **Shard parameters (EDIT THIS BEFORE EACH RUN)**

Set `SHARD_IDX` to 0, 1, or 2.

Three shards together cover ~60 k windows (=500 h audio) of LibriSpeech
`train-clean-360` (104 014 utterances) + `train-other-500` (148 688
utterances). Combined ~860 h. We allocate disjoint utterance ranges per shard
so no audio is repeated across shards.

| Shard | Source | Utterance range (skip..skip+take) | Target windows |
|---|---|---|---|
| 0 | train-clean-360 | 0..52 000 | 20 000 |
| 1 | train-clean-360 | 52 000..104 014, then train-other-500 0..30 000 | 20 000 |
| 2 | train-other-500 | 30 000..148 688 | 20 000 |


In [17]:
import pathlib

# ════════════════════════════════════════════════════════════════════════
# EDIT THESE TWO LINES EACH SESSION
SHARD_IDX = 2    # 0, 1, or 2
PART_IDX  = 3    # 0, 1, 2, or 3  (each part = 5k windows)
# ════════════════════════════════════════════════════════════════════════

assert SHARD_IDX in (0, 1, 2)
assert PART_IDX  in (0, 1, 2, 3)

WINDOWS_PER_PART   = 5_000
WINDOW_OFFSET      = PART_IDX * WINDOWS_PER_PART
WINDOW_SECONDS     = 30
TARGET_SR          = 24_000
SAMPLES_PER_WINDOW = TARGET_SR * WINDOW_SECONDS  # 720_000
T_FRAMES           = 375
TOPK               = 256

# Audio slice: ~12k flac files per part at LibriSpeech avg length
FLAC_PER_PART = 12_500
FLAC_SKIP     = PART_IDX * FLAC_PER_PART

SHARD_PLAN = {
    0: [("clean", "train.360", FLAC_SKIP, FLAC_PER_PART + 3_000)],
    1: [("clean", "train.360", 52_000 + FLAC_SKIP, FLAC_PER_PART + 3_000),
        ("other", "train.500", max(0, FLAC_SKIP - 52_000), FLAC_PER_PART + 3_000)],
    2: [("other", "train.500", 30_000 + FLAC_SKIP, FLAC_PER_PART + 3_000)],
}

# Output dir for this part — lives in /kaggle/working (19.5 GB cap, 18.24 GB needed)
OUT_DIR = pathlib.Path(f"/kaggle/working/cache_s{SHARD_IDX}p{PART_IDX}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset slug — all 4 parts of a shard go to the same dataset as new versions
DATASET_SLUG = f"mhassann/moshi-cache-shard{SHARD_IDX}"

print(f"=== SHARD {SHARD_IDX}, PART {PART_IDX} ===")
print(f"Window offset  : {WINDOW_OFFSET} → {WINDOW_OFFSET + WINDOWS_PER_PART - 1}")
print(f"Flac skip      : {FLAC_SKIP}")
print(f"Output dir     : {OUT_DIR}")
print(f"Dataset slug   : {DATASET_SLUG}")
import os
stat = os.statvfs("/kaggle/working")
print(f"/kaggle/working free: {stat.f_bavail * stat.f_frsize / 1e9:.1f} GB")


=== SHARD 2, PART 3 ===
Window offset  : 15000 → 19999
Flac skip      : 37500
Output dir     : /kaggle/working/cache_s2p3
Dataset slug   : mhassann/moshi-cache-shard2
/kaggle/working free: 20.9 GB


## Cell 6+7+8 — Audio → LMDB in one pass (no intermediate file)

Reads LibriSpeech flac files, runs teacher inference, and writes directly
to LMDB. No bin/npy intermediate file — avoids the 57.6 GB disk spike.


In [18]:
import pathlib, time, gc, json
import numpy as np
import soundfile as sf
import soxr
import torch
import os

# ── Audio source paths ────────────────────────────────────────────────────────
KNOWN_PATHS = {
    "train-clean-100": next(
        (p for p in [
            pathlib.Path("/kaggle/input/datasets/mhassann/librispeech-train-clean-100/LibriSpeech/train-clean-100"),
            pathlib.Path("/kaggle/input/datasets/tasfiatanha/librispeech-train-clean-100/LibriSpeech/train-clean-100"),
            pathlib.Path("/kaggle/input/librispeech-train-clean-100/LibriSpeech/train-clean-100"),
        ] if p.exists()),
        pathlib.Path("/kaggle/input/datasets/mhassann/librispeech-train-clean-100/LibriSpeech/train-clean-100")
    ),
    "train-clean-360": pathlib.Path("/kaggle/input/datasets/manancodes/librispeech-train-clean-360/LibriSpeech/train-clean-360"),
    "train-other-500": pathlib.Path("/kaggle/input/datasets/fredrelec/train-other-500/LibriSpeech/train-other-500"),
}
SUBSET_KEY_MAP = {
    ("clean", "train.360"): "train-clean-360",
    ("other", "train.500"): "train-other-500",
    ("clean", "train.100"): "train-clean-100",
}
for name, path in KNOWN_PATHS.items():
    print(f"  {'OK' if path.exists() else 'MISSING'}  {name}")

# ── Output memmap files ───────────────────────────────────────────────────────
# Exact pre-allocated size — no sparse files, no LMDB, no compaction needed.
# hidden:   (5000, 375, 4096) float16 = 15.36 GB
# topk_idx: (5000, 375, 256)  int32   =  1.92 GB
# topk_val: (5000, 375, 256)  float16 =  0.96 GB
# Total: 18.24 GB exact

H_PATH   = OUT_DIR / "hidden.npy"
IDX_PATH = OUT_DIR / "topk_idx.npy"
VAL_PATH = OUT_DIR / "topk_val.npy"

# Check if partially done (resume support)
def count_done(path, shape):
    if not path.exists(): return 0
    try:
        mm = np.memmap(path, dtype='float16' if 'hidden' in str(path) or 'val' in str(path) else 'int32',
                       mode='r', shape=shape)
        # Count non-zero rows (zero = not yet written)
        done = int(np.sum(mm[:, 0, 0] != 0))
        del mm
        return done
    except: return 0

already_done = count_done(H_PATH, (WINDOWS_PER_PART, T_FRAMES, 4096))
print(f"\nAlready written: {already_done}/{WINDOWS_PER_PART} windows")

if already_done >= WINDOWS_PER_PART:
    print("All windows already written — skipping cache loop")
    n_done = WINDOWS_PER_PART
else:
    # Create or open memmaps
    mode = 'r+' if H_PATH.exists() else 'w+'
    h_mm   = np.memmap(H_PATH,   dtype='float16', mode=mode, shape=(WINDOWS_PER_PART, T_FRAMES, 4096))
    idx_mm = np.memmap(IDX_PATH, dtype='int32',   mode=mode, shape=(WINDOWS_PER_PART, T_FRAMES, TOPK))
    val_mm = np.memmap(VAL_PATH, dtype='float16', mode=mode, shape=(WINDOWS_PER_PART, T_FRAMES, TOPK))

    stat = os.statvfs("/kaggle/working")
    print(f"/kaggle/working free after alloc: {stat.f_bavail * stat.f_frsize / 1e9:.1f} GB")

    # ── Moshi inference setup ─────────────────────────────────────────────────
    NUM_CB_TEACHER = teacher_lm.num_codebooks
    assert NUM_CB_TEACHER == 17
    TEXT_PAD      = 3
    MIMI_AUDIO_CB = 8
    batch_codes = torch.full((1, NUM_CB_TEACHER, T_FRAMES),
                             TEXT_PAD, dtype=torch.long, device="cuda:0")

    # ── Cache loop ────────────────────────────────────────────────────────────
    print(f"\nCache loop: writing windows {WINDOW_OFFSET} → {WINDOW_OFFSET + WINDOWS_PER_PART - 1}")
    print(f"Resuming from local index {already_done}")

    buf    = np.zeros(0, dtype=np.float32)
    n_done = already_done
    n_skip = 0
    t_start = time.time()

    for subset, split, skip, take in SHARD_PLAN[SHARD_IDX]:
        if n_done >= WINDOWS_PER_PART: break
        local_name = SUBSET_KEY_MAP.get((subset, split))
        local_path = KNOWN_PATHS.get(local_name) if local_name else None

        if not (local_path and local_path.exists()):
            print(f"  MISSING: {local_name} — skipping"); continue

        print(f"\nLocal {local_name}: skip={skip} take={take}")
        flac_files = sorted(local_path.rglob("*.flac"))[skip: skip + take]
        local_n = 0

        for fpath in flac_files:
            if n_done >= WINDOWS_PER_PART: break
            local_n += 1
            try:
                arr, sr = sf.read(str(fpath), dtype="float32")
                if arr.ndim > 1: arr = arr.mean(axis=1).astype(np.float32)
                if sr != TARGET_SR:
                    arr = soxr.resample(arr, sr, TARGET_SR, quality="HQ").astype(np.float32)
                buf = np.concatenate([buf, arr])

                while buf.shape[0] >= SAMPLES_PER_WINDOW and n_done < WINDOWS_PER_PART:
                    wav_np = buf[:SAMPLES_PER_WINDOW]
                    buf    = buf[SAMPLES_PER_WINDOW:]

                    # Skip already-written windows (resume)
                    local_idx = n_done  # index within this part's 5000
                    if local_idx < already_done:
                        n_done += 1
                        continue

                    try:
                        wav_t = torch.from_numpy(wav_np.copy()).to(
                            device="cuda:0", dtype=torch.float16).unsqueeze(0).unsqueeze(0)
                        with torch.inference_mode():
                            codes = mimi.encode(wav_t)
                            batch_codes.fill_(TEXT_PAD)
                            batch_codes[:, 1:1+MIMI_AUDIO_CB, :] = codes.to(torch.long)
                            hidden, text_logits = teacher_lm.forward_text(batch_codes)
                            text_logits = text_logits.squeeze(0).squeeze(0)
                            topk_val, topk_idx = text_logits.topk(TOPK, dim=-1)

                        h_mm[local_idx]   = hidden.squeeze(0).to("cpu", dtype=torch.float16).numpy()
                        idx_mm[local_idx] = topk_idx.to("cpu", dtype=torch.int32).numpy()
                        val_mm[local_idx] = topk_val.to("cpu", dtype=torch.float16).numpy()
                        n_done += 1

                        # Flush every 100 windows to persist to disk
                        if n_done % 100 == 0:
                            h_mm.flush(); idx_mm.flush(); val_mm.flush()

                    except RuntimeError as e:
                        n_skip += 1
                        print(f"  window {WINDOW_OFFSET+n_done} FAILED: {e}")
                        torch.cuda.empty_cache()
                        n_done += 1  # skip this slot

            except Exception as e:
                print(f"  skip {fpath.name}: {e}")

            if local_n % 500 == 0 or n_done >= WINDOWS_PER_PART:
                elapsed = time.time() - t_start
                wpm = (n_done - already_done) / max(elapsed, 1) * 60
                eta = (WINDOWS_PER_PART - n_done) / max(wpm/60, 1e-6) / 60
                print(f"  [{local_n}/{len(flac_files)}] {n_done}/{WINDOWS_PER_PART} windows  "
                      f"wpm={wpm:.1f}  eta={eta:.0f}min  "
                      f"gpu0={torch.cuda.mem_get_info(0)[0]/1e9:.1f}GB")

    # Final flush
    h_mm.flush(); idx_mm.flush(); val_mm.flush()
    del h_mm, idx_mm, val_mm
    gc.collect()

    elapsed = time.time() - t_start
    print(f"\nDone: {n_done} windows, {n_skip} skipped, {elapsed/60:.1f} min")

# ── Save throughput log ───────────────────────────────────────────────────────
log = {
    "shard_idx": SHARD_IDX, "part_idx": PART_IDX,
    "window_offset": WINDOW_OFFSET, "n_windows": n_done,
    "windows_per_minute": round((n_done-already_done) / max(time.time()-t_start, 1) * 60, 1)
    if already_done < WINDOWS_PER_PART else 0,
}
(OUT_DIR / "throughput.json").write_text(json.dumps(log, indent=2))

# Verify files
for p, shape, dt in [
    (H_PATH,   (WINDOWS_PER_PART, T_FRAMES, 4096), 'float16'),
    (IDX_PATH, (WINDOWS_PER_PART, T_FRAMES, TOPK),  'int32'),
    (VAL_PATH, (WINDOWS_PER_PART, T_FRAMES, TOPK),  'float16'),
]:
    assert p.exists(), f"Missing: {p.name}"
    sz = p.stat().st_size / 1e9
    print(f"  {p.name:<15} {sz:.2f} GB")

stat = os.statvfs("/kaggle/working")
print(f"\n/kaggle/working free: {stat.f_bavail * stat.f_frsize / 1e9:.1f} GB")
print(f"=== Cell 6+7+8 PASSED ({n_done} windows) ===")


  OK  train-clean-100
  OK  train-clean-360
  OK  train-other-500

Already written: 0/5000 windows
/kaggle/working free after alloc: 20.9 GB

Cache loop: writing windows 15000 → 19999
Resuming from local index 0

Local train-other-500: skip=67500 take=15500
  [500/15500] 205/5000 windows  wpm=41.1  eta=117min  gpu0=7.5GB
  [1000/15500] 412/5000 windows  wpm=66.6  eta=69min  gpu0=7.5GB
  [1500/15500] 609/5000 windows  wpm=82.9  eta=53min  gpu0=7.5GB
  [2000/15500] 798/5000 windows  wpm=94.2  eta=45min  gpu0=7.5GB
  [2500/15500] 1004/5000 windows  wpm=103.5  eta=39min  gpu0=7.5GB
  [3000/15500] 1206/5000 windows  wpm=110.8  eta=34min  gpu0=7.5GB
  [3500/15500] 1381/5000 windows  wpm=115.7  eta=31min  gpu0=7.5GB
  [4000/15500] 1583/5000 windows  wpm=120.4  eta=28min  gpu0=7.5GB
  [4500/15500] 1797/5000 windows  wpm=124.8  eta=26min  gpu0=7.5GB
  [5000/15500] 2014/5000 windows  wpm=128.3  eta=23min  gpu0=7.5GB
  [5500/15500] 2219/5000 windows  wpm=130.9  eta=21min  gpu0=7.5GB
  [6000/15500

## ~~Cell merged above — skip~~


In [ ]:
print("This cell merged into Cell 6+7+8 above — skip")


## ~~Cell merged above — skip~~


In [ ]:
print("This cell merged into Cell 6+7+8 above — skip")


## Cell 9 — Final compact + tar + MANIFEST

Even if mid-loop compacts ran, do one final compact-and-tar on the live env so
the upload tar exactly matches the LMDB at session end (no race with the
incremental snapshot). Then drop the live `shard{N}.lmdb` dir to free
/kaggle/working space for the upload itself.


In [19]:
import json, subprocess, pathlib
import torch, transformers

# No compaction needed — numpy memmaps are already exact-size files.
# Just write the manifest and we're done.

log = json.loads((OUT_DIR / "throughput.json").read_text())

env_out  = subprocess.run(["pip", "freeze"],    capture_output=True, text=True).stdout
nvid_out = subprocess.run(["nvidia-smi", "-q"], capture_output=True, text=True).stdout
(OUT_DIR / "env.txt").write_text(env_out + "\n=== nvidia-smi ===\n" + nvid_out)

(OUT_DIR / "MANIFEST.md").write_text(f"""# MANIFEST — moshi-cache-shard{SHARD_IDX} part {PART_IDX}

| Key | Value |
|---|---|
| shard_idx | {SHARD_IDX} |
| part_idx | {PART_IDX} |
| window_offset | {WINDOW_OFFSET} |
| n_windows | {log['n_windows']} |
| windows_per_minute | {log['windows_per_minute']} |
| torch | {torch.__version__} |

## Files
- `hidden.npy`   : float16 ({WINDOWS_PER_PART}, {T_FRAMES}, 4096)
- `topk_idx.npy` : int32   ({WINDOWS_PER_PART}, {T_FRAMES}, {TOPK})
- `topk_val.npy` : float16 ({WINDOWS_PER_PART}, {T_FRAMES}, {TOPK})

## Load in Phase-1
```python
import numpy as np
h   = np.memmap("/kaggle/input/moshi-cache-shard{SHARD_IDX}/hidden.npy",
                dtype="float16", mode="r", shape=({WINDOWS_PER_PART}, {T_FRAMES}, 4096))
idx = np.memmap("/kaggle/input/moshi-cache-shard{SHARD_IDX}/topk_idx.npy",
                dtype="int32",   mode="r", shape=({WINDOWS_PER_PART}, {T_FRAMES}, {TOPK}))
val = np.memmap("/kaggle/input/moshi-cache-shard{SHARD_IDX}/topk_val.npy",
                dtype="float16", mode="r", shape=({WINDOWS_PER_PART}, {T_FRAMES}, {TOPK}))
```
""")

print("Files in OUT_DIR:")
total_gb = 0
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        gb = p.stat().st_size / 1e9
        total_gb += gb
        print(f"  {p.name:<20} {gb:.2f} GB")
print(f"  TOTAL               {total_gb:.2f} GB")
print("=== Cell 9 PASSED ===")


Files in OUT_DIR:
  MANIFEST.md          0.00 GB
  env.txt              0.00 GB
  hidden.npy           15.36 GB
  throughput.json      0.00 GB
  topk_idx.npy         1.92 GB
  topk_val.npy         0.96 GB
  TOTAL               18.24 GB
=== Cell 9 PASSED ===


## Cell 10 — Push `moshi-teacher-cache-shard{SHARD_IDX}`

This time we're pushing a single tar file (not a directory), so no
`--dir-mode` quirks. `kaggle datasets create` succeeds first run for
shard 0/1/2 because each id is distinct.


In [21]:
import subprocess, json, pathlib, os

username   = os.environ.get("KAGGLE_USERNAME", "mhassann")
# Each part is its own dataset — simple, independent, no growing tars
dataset_id = f"{username}/moshi-cache-s{SHARD_IDX}p{PART_IDX}"

metadata = {
    "title":    f"moshi-cache-s{SHARD_IDX}p{PART_IDX}",
    "id":       dataset_id,
    "licenses": [{"name": "CC0-1.0"}],
}
(OUT_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

print(f"Uploading {OUT_DIR} → {dataset_id}")
print(f"Windows {WINDOW_OFFSET} → {WINDOW_OFFSET + WINDOWS_PER_PART - 1}")

# Always create (each part is a new dataset)
print("Creating dataset ...")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(OUT_DIR)],
    capture_output=True, text=True)
print(r.stdout or "(no stdout)")
if r.returncode != 0:
    # Dataset exists — version bump
    print("Dataset exists, version bumping ...")
    r2 = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(OUT_DIR),
         "-m", f"windows {WINDOW_OFFSET}-{WINDOW_OFFSET+WINDOWS_PER_PART-1}"],
        capture_output=True, text=True)
    print(r2.stdout or "(no stdout)")
    if r2.returncode != 0: print("STDERR:", r2.stderr)
    else: print(f"SUCCESS — kaggle.com/{dataset_id}")
else:
    print(f"SUCCESS — kaggle.com/{dataset_id}")

print(f"\n=== SHARD {SHARD_IDX} PART {PART_IDX} COMPLETE ===")
print(f"Dataset: {dataset_id}")
print(f"Next: set PART_IDX={PART_IDX+1} (no restore needed — parts are independent)")


Uploading /kaggle/working/cache_s2p3 → mhassann/moshi-cache-s2p3
Windows 15000 → 19999
Creating dataset ...
Starting upload for file topk_idx.npy
Upload successful: topk_idx.npy (2GB)
Starting upload for file hidden.npy
Upload successful: hidden.npy (14GB)
Starting upload for file env.txt
Upload successful: env.txt (38KB)
Starting upload for file topk_val.npy
Upload successful: topk_val.npy (916MB)
Starting upload for file throughput.json
Upload successful: throughput.json (115B)
Starting upload for file MANIFEST.md
Upload successful: MANIFEST.md (797B)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/mhassann/moshi-cache-s2p3

SUCCESS — kaggle.com/mhassann/moshi-cache-s2p3

=== SHARD 2 PART 3 COMPLETE ===
Dataset: mhassann/moshi-cache-s2p3
Next: set PART_IDX=4 (no restore needed — parts are independent)


In [16]:
!rm -rf /kaggle/working/*



In [22]:
# Quick verification — all 12 datasets should exist
import subprocess
for shard in range(3):
    for part in range(4):
        slug = f"mhassann/moshi-cache-s{shard}p{part}"
        r = subprocess.run(["kaggle", "datasets", "list", "--search", f"moshi-cache-s{shard}p{part}"],
                          capture_output=True, text=True)
        status = "OK" if slug.split("/")[1] in r.stdout else "MISSING"
        print(f"  {status}  {slug}")

  OK  mhassann/moshi-cache-s0p0
  OK  mhassann/moshi-cache-s0p1
  OK  mhassann/moshi-cache-s0p2
  OK  mhassann/moshi-cache-s0p3
  OK  mhassann/moshi-cache-s1p0
  OK  mhassann/moshi-cache-s1p1
  OK  mhassann/moshi-cache-s1p2
  OK  mhassann/moshi-cache-s1p3
  OK  mhassann/moshi-cache-s2p0
  OK  mhassann/moshi-cache-s2p1
  OK  mhassann/moshi-cache-s2p2
  OK  mhassann/moshi-cache-s2p3
